# Advanced RNN and transfer learning for NLP; course capstone — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D025_04_Transfer_Learning_NLP.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Transfer Learning with Pre-trained Embeddings (GloVe)

## What is Transfer Learning in NLP?
Transfer learning involves taking a model (or parts of a model) trained on a large dataset and applying it to a different but related task. 

In the context of processing text with RNNs, the most common form of transfer learning is using **Pre-trained Word Embeddings**.

Instead of learning the meaning of words from scratch (which requires a huge amount of data), we can initialize our `Embedding` layer with vectors learned from massive corpuses (like Wikipedia).

## GloVe (Global Vectors for Word Representation)
GloVe is a popular set of pre-trained word vectors. We will use the 100-dimensional vectors trained on Wikipedia.

In [ ]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.random.set_seed(42)

In [ ]:
# ═══ CHECKPOINT — "Downloading GloVe Embeddings" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print("Checkpoint restored. Ready for: Downloading GloVe Embeddings")

## 1. Downloading GloVe Embeddings
We will download the GloVe dataset. Note: This file is around 800MB.

> **No download needed.** The original lesson fetched the full 822 MB GloVe archive here. This copy ships `glove.6B.100d.imdb10k.txt`: the same 100-dimensional GloVe vectors, kept only for the 9,999 IMDB words this lesson indexes (9,796 of them have a vector). Words outside it stay at zero in the embedding matrix, exactly as they would with the full file.

## 2. Preparing the Data (IMDB)

In [ ]:
# TODO: compute `VOCAB_SIZE`, `MAX_LEN`, `X_train`
# Name your result: VOCAB_SIZE, MAX_LEN, X_train, X_test

## 3. Creating the Embedding Matrix
We need to create a matrix where row `i` contains the GloVe vector for the word with index `i`.

In [ ]:
# TODO: compute `embeddings_index`, `f`, `values`
# Name your result: embeddings_index, f, values, word

## 4. Building the Model with Pre-trained Embeddings
We use the `Embedding` layer but initialize it with our `embedding_matrix` and set `trainable=False` to keep the pre-trained weights frozen.

In [ ]:
# TODO: compute `model`, `weights`, `input_length` from `accuracy`
# Name your result: model, weights, input_length, trainable

In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

In [ ]:
# ═══ CHECKPOINT — "Fine-Tuning" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
VOCAB_SIZE = 10000
MAX_LEN = 200
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
X_train = pad_sequences(X_train, maxlen=MAX_LEN)
X_test = pad_sequences(X_test, maxlen=MAX_LEN)
word_index = imdb.get_word_index()
embeddings_index = {}
f = open('../../00-data/glove.6B.100d.imdb10k.txt', encoding='utf-8')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
embedding_dim = 100
embedding_matrix = np.zeros((VOCAB_SIZE, embedding_dim))
for word, i in word_index.items():
    if i < VOCAB_SIZE:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
model = Sequential()
model.add(Embedding(VOCAB_SIZE, embedding_dim, weights=[embedding_matrix], input_length=MAX_LEN, trainable=False))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=1, batch_size=64, validation_split=0.2)

print("Checkpoint restored. Ready for: Fine-Tuning")

## 5. Fine-Tuning
After training the top layers, we can unfreeze the embedding layer to fine-tune the word vectors for our specific task.

In [ ]:
# TODO: compute `history_fine` from `accuracy`
# Name your result: history_fine